# Part 3 — Review Summarisation with Generative AI

## Stage 1: the evidence pack

The articles must contain factual claims — which products lead a category, what
people complain about, how they differ. If we hand raw reviews to a language
model and ask it to decide those things, it will produce confident, plausible,
partly-invented answers that are very hard to audit afterwards.

So we split the task:

1. **Compute the facts** (this notebook section) — rankings, complaint themes,
   and verbatim quotes, derived from the clustered data.
2. **Write the prose** (Stage 2) — the model phrases what we already computed
   and is given nothing else.

The output is `evidence_pack.json`: every cluster with **all** its rankable
products ranked, so the article format remains an open choice (top-N, buyer's
guide by use case, complaint round-up, etc.) rather than being baked in here.

In [1]:
# cell 1: Imports and configuration
import json
import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

DATA = Path("../data/processed")
OUT  = DATA / "evidence_pack.json"

MIN_REVIEWS     = 15   # minimum reviews for a product to be ranked
PRIOR_WEIGHT    = 50   # shrinkage strength (~"50 phantom average reviews")
N_QUOTES        = 3    # verbatim quotes kept per product, per sentiment
MAX_QUOTE_CHARS = 300

# Excluded for data quality — recorded in the pack so it is visible, not silent.
EXCLUDE_ASINS = {
    "B00QJDU3KY": "corrupt category string ('mazon.co.uk,Amazon Devices') — "
                  "a Kindle Paperwhite misclustered into Echo/Fire TV",
}

## 1.1 Resolving product names

Products must be named in the articles, but the `name` column cannot be trusted:
it is 19.5% null and **misaligned with `asins`** — an Accessories product is
labelled "All-New Kindle E-reader", a Fire Tablet is labelled "Echo (White)".

`keys` is null-free, unique per ASIN, and always agreed with the product's own
categories on inspection. It holds slugs (`firetvstreamingmediaplayer2015model`),
so we segment them back into words with dynamic programming — greedy matching
splits words across real boundaries (`glarefreetouch` → "glare freet ouch").

In [2]:
# cell 2: name resolution (so it keeps "keys" as actual name)
DOMAIN = {
    "all","new","amazon","kindle","fire","tablet","tablets","paperwhite","voyage",
    "oasis","echo","dot","tap","show","streaming","media","player","stick","tv",
    "hd","hdx","display","wifi","gb","includes","special","offers","black","white",
    "blue","green","red","magenta","tangerine","silver","aluminum","cayenne","kids",
    "edition","proof","case","cover","standing","protective","leather","lighted",
    "charger","power","fast","adapter","official","oem","usb","micro","cable",
    "cables","angle","generation","release","model","ereader","ereaders","reader",
    "readers","ebook","glare","free","touch","screen","wireless","bluetooth",
    "speaker","portable","works","with","most","ips","brand","inch","resolution",
    "high","card","slot","hours","battery","life","refurbished","certified",
    "monochrome",
}
# NB: no two-letter fragments — with the domain bonus, "wi"+"fi" would beat "wifi".

_sys = Path("/usr/share/dict/words")
DICT = ({w.strip().lower() for w in _sys.read_text(errors="ignore").splitlines()
         if len(w.strip()) >= 3 and w.strip().isalpha()} if _sys.exists() else set())
DICT |= DOMAIN
MAX_WORD = max(len(w) for w in DICT)


def split_slug(slug):
    """Segment a concatenated slug into words (DP over all segmentations)."""
    n = len(slug)
    best = [(-1e18, 0, "")] * (n + 1)
    best[0] = (0.0, 0, "")
    for i in range(1, n + 1):
        for j in range(max(0, i - MAX_WORD), i):
            if best[j][0] <= -1e17:
                continue
            piece = slug[j:i]
            if piece.isdigit():   gain = len(piece) ** 2
            elif piece in DOMAIN: gain = len(piece) ** 2 + 60   # prefer product terms
            elif piece in DICT:   gain = len(piece) ** 2
            else:                 gain = -18 * len(piece)       # penalise gibberish
            if best[j][0] + gain > best[i][0]:
                best[i] = (best[j][0] + gain, j, piece)
    words, i = [], n
    while i > 0:
        _, j, piece = best[i]; words.append(piece); i = j
    out = []
    for tok in reversed(words):
        if out and len(tok) == 1 and not tok.isdigit() and len(out[-1]) <= 2:
            out[-1] += tok
        else:
            out.append(tok)
    return " ".join(out).title()


def modal_name(series):
    vals = series.dropna()
    if vals.empty:
        return None
    best = vals.mode().iloc[0].replace("\r", "\n").split("\n")[0]
    return re.sub(r"(,\s*)+$", "", best).strip() or None


def name_from_keys(keys):
    parts = [p.split("/")[0] for p in str(keys).split(",")]
    parts = [p for p in parts if re.search(r"[a-z]{6}", p)]
    if not parts:
        return None
    text = split_slug(max(parts, key=len))
    text = re.sub(r"\bB\s*\d[\dA-Za-z ]*$", "", text).strip()    # trailing ASIN
    text = re.sub(r"\b(\w+)( \1\b)+", r"\1", text, flags=re.I)   # "Amazon Amazon"
    return re.sub(r"\s{2,}", " ", text).strip() or None


# sanity check
for s in ["certifiedrefurbishedamazonfiretv", "allnewkindleereaderblack6glarefreetouchscreen"]:
    print(split_slug(s))

Certified Refurbished Amazon Fire Tv
All New Kindle Ereader Black 6 Glare Free Touch Screen


## 1.2 Product table, names and ranking

Products are ranked by a **shrunken mean** (Bayesian average), not the raw mean:

$$\text{score} = \frac{n \cdot \bar{r} + m \cdot C}{n + m}$$

where $C$ is the global mean rating and $m$ = 50. Without it a product with 16
reviews outranks one with 401 on noise alone.

In [3]:
# cell 3: Product table 
reviews  = pd.read_csv(DATA / "reviews_with_clusters.csv")
products = pd.read_csv(DATA / "product_clusters.csv")
assert reviews["cluster"].notna().all(), "unclustered reviews present"

# --- names: `keys` first, `name` only as a last resort -----------------------
names = reviews.groupby("asins").agg(name_raw=("name", modal_name), keys=("keys", "first"))
names["from_keys"]   = [name_from_keys(r.keys) for r in names.itertuples()]
names["display_name"] = [r.from_keys or r.name_raw for r in names.itertuples()]
names["name_source"]  = ["keys" if r.from_keys else "name" for r in names.itertuples()]

products = products.merge(names[["display_name", "name_source"]],
                          left_on="asins", right_index=True, how="left")
assert products["display_name"].notna().all(), "a product has no resolvable name"

# --- data-quality exclusions -------------------------------------------------
excluded = products[products["asins"].isin(EXCLUDE_ASINS)]
products = products[~products["asins"].isin(EXCLUDE_ASINS)].copy()
reviews  = reviews[~reviews["asins"].isin(EXCLUDE_ASINS)].copy()

# --- disambiguate identical names (two ASINs can share a `keys` slug) ---------
dup = products["display_name"].duplicated(keep=False)
products.loc[dup, "display_name"] += " [" + products.loc[dup, "asins"] + "]"
assert products["display_name"].is_unique, "display names still collide"

# --- shrunken score ----------------------------------------------------------
prior = products["mean_rating"].mean()
products["score"] = ((products["n_reviews"] * products["mean_rating"] + PRIOR_WEIGHT * prior)
                     / (products["n_reviews"] + PRIOR_WEIGHT))
products["rankable"] = products["n_reviews"] >= MIN_REVIEWS

print(f"{len(products)} products | excluded: {list(EXCLUDE_ASINS)} | prior C={prior:.3f}")
products.sort_values("score", ascending=False).head()

37 products | excluded: ['B00QJDU3KY'] | prior C=4.405


,asins,n_reviews,categories,brand,mean_rating,cluster,cluster_name,rankable,display_name,name_source,score
11,B00OQVZDJM,3176,"Walmart for Business,Office Electronics,Tablet...",Amazon,4.772355,3,Kindle E-Readers,True,Amazon Kindle Paperwhite Ebook Reader 4 Gb 6 M...,keys,4.766654
17,B00U3FPN4U,5056,"Back To College,College Electronics,College Tv...",Amazon Fire Tv,4.707278,2,"Echo, Fire TV & Smart Home",True,Fire Tv Streaming Media Player 2015 Model,keys,4.704314
6,B00IOY8XWQ,580,"Walmart for Business,Office Electronics,Tablet...",Amazon,4.729310,3,Kindle E-Readers,True,Kindle Voyage Ereader 6 High Resolution Displa...,keys,4.703534
8,"B00L9EPT8O,B01E6AO69U",6619,"Stereos,Remote Controls,Amazon Echo,Audio Dock...",Amazon,4.671098,2,"Echo, Fire TV & Smart Home",True,Echo White,keys,4.669100
7,B00IOYAM4I,51,"Computers & Tablets,E-Readers & Accessories,eB...",Amazon,4.862745,3,Kindle E-Readers,True,Amazon Kindle Voyage 4 Gb Wifi 3 Gb Lack,keys,4.635904


In [4]:
# cell 4: themes & quotes
import numpy as np

# Product names and filler that appear regardless of sentiment — they cannot
# distinguish anything, so they are removed from both directions.
BASE_STOP = {"product","amazon","kindle","fire","tablet","tablets","echo","just","did","does",
             "don","doesn","didn","isn","wasn","couldn","wouldn","won","really","got","bought",
             "buy","purchased","use","used","using","like","would","could","will","one","get",
             "even","back","also","much","make","way","thing","things","ipad","lot","kept","pay"}

# Generic enthusiasm: tells you something is good, but not WHAT is good.
# Measured: without this filter every cluster's "loved" list is the same four
# words — "great, love, easy, good" — and so is every product's within a cluster.
# Removing them surfaces the substance: price, reading, battery, music, kids.
PRAISE_STOP = {"great","love","loves","loved","easy","good","nice","fine","perfect","works",
               "work","best","awesome","excellent","happy","recommend","pleased","amazing",
               "wonderful","glad","satisfied","better","well","super","pretty","very"}

# Kept for backwards compatibility with the old name used elsewhere.
COMPLAINT_STOP = BASE_STOP


def distinctive_themes(focus_texts, contrast_texts, stop, top_k=6):
    """Terms characteristic of `focus` reviews *relative to* `contrast` reviews.

    Raw frequency does not work in either direction. Negative reviews still say
    "good" and "great", so counting words in them lists those as complaints.
    The fix is to ask what is common in one group but NOT the other.
    """
    focus    = [t for t in focus_texts.dropna().astype(str)][:600]
    contrast = [t for t in contrast_texts.dropna().astype(str)][:600]
    if len(focus) < 5 or len(contrast) < 5:
        return []
    docs = focus + contrast
    y = np.array([1] * len(focus) + [0] * len(contrast))
    try:
        tf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3,
                             max_df=0.6, token_pattern=r"[a-z]{3,}", sublinear_tf=True)
        X = tf.fit_transform([d.lower() for d in docs])
    except ValueError:
        return []
    diff = pd.Series(np.asarray(X[y == 1].mean(0)).ravel() - np.asarray(X[y == 0].mean(0)).ravel(),
                     index=tf.get_feature_names_out())
    keep = [t for t in diff.index if not any(w in stop for w in t.split())]
    return diff[keep].sort_values(ascending=False).head(top_k).index.tolist()


def complaint_themes(neg_texts, pos_texts, top_k=6):
    """What goes wrong: distinctive to negative reviews."""
    return distinctive_themes(neg_texts, pos_texts, BASE_STOP, top_k)


def loved_themes(pos_texts, neg_texts, top_k=6):
    """What goes right: distinctive to positive reviews, minus generic praise."""
    return distinctive_themes(pos_texts, neg_texts, BASE_STOP | PRAISE_STOP, top_k)


def short_name(n, max_words=9):
    """Trim marketing boilerplate. Callers must check for collisions."""
    return " ".join(re.split(r"\b(?:Includes|With Special)\b", n)[0].strip().split()[:max_words])


def pick_quotes(sub, sentiment, n=N_QUOTES):
    q = sub[sub["sentiment"] == sentiment].copy()
    if q.empty:
        return []
    q["_len"] = q["reviews.text"].astype(str).str.len()
    q = q[q["_len"].between(60, MAX_QUOTE_CHARS)]
    if q.empty:
        return []
    q = q.sort_values(["reviews.numHelpful", "_len"], ascending=[False, False])
    return [{"rating": float(r["reviews.rating"]),
             "helpful": int(r["reviews.numHelpful"]) if pd.notna(r["reviews.numHelpful"]) else 0,
             "text": str(r["reviews.text"]).strip()}
            for _, r in q.head(n).iterrows()]


# sanity check: praise filter should give substance, not enthusiasm
_c = reviews[reviews["cluster_name"] == "Echo, Fire TV & Smart Home"]
print("loved   :", loved_themes(_c[_c.sentiment == "Positive"]["reviews.text"],
                                _c[_c.sentiment == "Negative"]["reviews.text"]))
print("criticised:", complaint_themes(_c[_c.sentiment == "Negative"]["reviews.text"],
                                      _c[_c.sentiment == "Positive"]["reviews.text"]))

loved   : ['music', 'alexa', 'fun', 'sound', 'gift', 'family']
criticised: ['returned', 'work', 'money', 'item', 'return', 'connection']


## 1.3 Assemble the pack

Every rankable product is included with its rank — **no article format is
assumed**. `data_notes` carries the caveats the writer must respect, the most
important being that ratings are heavily skewed positive (~2% negative overall),
so most clusters have no genuinely bad product to warn about.

In [5]:
# cell 5: assemble

pack = {
    "meta": {
        "n_products": int(len(products)),
        "n_reviews": int(len(reviews)),
        "min_reviews_to_rank": MIN_REVIEWS,
        "prior_weight": PRIOR_WEIGHT,
        "global_mean_rating": round(float(prior), 3),
        "ranking_note": "Ranked by shrunken mean (Bayesian average), not raw mean.",
        "excluded_products": [{"asin": r.asins, "reason": EXCLUDE_ASINS[r.asins]}
                              for r in excluded.itertuples()],
    },
    "clusters": [],
}

for cname, prods in products.groupby("cluster_name"):
    cr = reviews[reviews["cluster_name"] == cname]
    ranked = prods[prods["rankable"]].sort_values("score", ascending=False)
    sent = cr["sentiment"].value_counts()

    blocks = []
    for rank, row in enumerate(ranked.itertuples(), 1):
        sub = cr[cr["asins"] == row.asins]
        neg = sub[sub["sentiment"] == "Negative"]
        pos = sub[sub["sentiment"] == "Positive"]
        blocks.append({
            "rank": rank,
            "asin": row.asins,
            "name": row.display_name,
            "n_reviews": int(row.n_reviews),
            "mean_rating": round(float(row.mean_rating), 2),
            "score": round(float(row.score), 3),
            "pct_positive": round(len(pos) / len(sub) * 100, 1),
            "pct_negative": round(len(neg) / len(sub) * 100, 1),
            "recommend_rate": (round(float(sub["reviews.doRecommend"].mean() * 100), 1)
                               if sub["reviews.doRecommend"].notna().any() else None),
            "categories": row.categories,
            # Both directions use the same method; below ~25 reviews on either
            # side the result is noise, so we return nothing rather than filler.
            "loved_themes": (loved_themes(pos["reviews.text"], neg["reviews.text"])
                             if len(neg) >= 25 else []),
            "complaint_themes": (complaint_themes(neg["reviews.text"], pos["reviews.text"])
                                 if len(neg) >= 25 else []),
            "quotes_positive": pick_quotes(sub, "Positive"),
            "quotes_negative": pick_quotes(sub, "Negative"),
        })

    lowest = ranked.iloc[-1] if len(ranked) else None
    cpos = cr[cr["sentiment"] == "Positive"]["reviews.text"]
    cneg = cr[cr["sentiment"] == "Negative"]["reviews.text"]
    pack["clusters"].append({
        "cluster_name": cname,
        "cluster_id": int(prods["cluster"].iloc[0]),
        "n_products": int(len(prods)),
        "n_rankable": int(len(ranked)),
        "n_reviews": int(len(cr)),
        "mean_rating": round(float(cr["reviews.rating"].mean()), 2),
        "sentiment_counts": {k: int(sent.get(k, 0)) for k in ["Positive", "Neutral", "Negative"]},
        "pct_positive": round(float(sent.get("Positive", 0) / len(cr) * 100), 1),
        "pct_negative": round(float(sent.get("Negative", 0) / len(cr) * 100), 1),
        "loved_themes": loved_themes(cpos, cneg, 8),
        "complaint_themes": complaint_themes(cneg, cpos, 8),
        "products": blocks,
        "data_notes": {
            "rating_spread": round(float(ranked["mean_rating"].max() - ranked["mean_rating"].min()), 2),
            "has_genuinely_bad_product": bool(lowest is not None and lowest.mean_rating < 4.0),
            "caution": ("A genuinely poor performer exists — criticism is supported."
                        if lowest is not None and lowest.mean_rating < 4.0 else
                        "All ranked products score well. Describe the last as 'lowest rated', "
                        "not as bad. Do not invent criticism."),
        },
    })

# Shorten names, but keep the full name wherever shortening would collide
# (truncation hides "16 GB" vs "32 GB" and turns two products into one).
for c in pack["clusters"]:
    shorts = [short_name(b["name"]) for b in c["products"]]
    dupes = {x for x in shorts if shorts.count(x) > 1}
    for b, sh in zip(c["products"], shorts):
        b["name"] = b["name"] if sh in dupes else sh
pack["clusters"].sort(key=lambda c: -c["n_reviews"])

for c in pack["clusters"]:
    print(f"\n{c['cluster_name']} — {c['n_products']} products "
          f"({c['n_rankable']} rankable), {c['n_reviews']:,} reviews, "
          f"{c['pct_positive']}% positive / {c['pct_negative']}% negative")
    print(f"   loved     : {', '.join(c['loved_themes'][:6])}")
    print(f"   criticised: {', '.join(c['complaint_themes'][:6])}")


Fire Tablets — 14 products (9 rankable), 17,627 reviews, 91.5% positive / 2.9% negative
   loved     : price, reading, size, read, books, fast
   criticised: slow, returned, charge, work, turn, return

Echo, Fire TV & Smart Home — 7 products (3 rankable), 12,330 reviews, 95.2% positive / 1.5% negative
   loved     : music, alexa, fun, sound, gift, family
   criticised: returned, work, money, item, return, connection

Kindle E-Readers — 6 products (5 rankable), 4,092 reviews, 96.9% positive / 1.0% negative
   loved     : reading, battery, sun, wife, able, life
   criticised: tried, work, customer, service, computer, usb

Accessories & Cables — 10 products (4 rankable), 560 reviews, 81.8% positive / 12.7% negative
   loved     : charges, needed, new, perfectly, thanks, item
   criticised: charger, included, expensive, charge, money, overpriced


# Stage 2 — Generating the articles

## What we measured about FLAN-T5-base

Before designing the generator we tested `google/flan-t5-base` (248M) on this data:

| Test | Result |
|---|---|
| Write a recommendation from a facts table | **Inverted the facts** — called a 4.59★ product "a really bad tablet" with "very poor screen quality" |
| Summarise positive reviews | Worked — "Good tablet for a really good price. Good parental controls." |
| Summarise negative reviews | **Copied source reviews verbatim** instead of summarising |

The positive result is partly an artifact: 1,537 near-identical positive reviews
force convergence on generic phrasing. With ~50 diverse negative reviews the
model has nothing to converge on and falls back to extraction.

## The resulting design

| Content | Produced by |
|---|---|
| Ratings, review counts, %negative, rankings, product names, quotes | **Template, straight from `pack`** — cannot be wrong |
| "What reviewers like" | **FLAN-T5 abstractive summarisation**, behind a copy-guard |

Any generated line is rejected if an 8-word sequence of it appears in a source
review — i.e. if the model extracted rather than summarised. Rejected lines are
omitted, never printed. The guard counter at the end reports how often the
pretrained model actually succeeded.

In [6]:
# cell 6: load model
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "google/flan-t5-base"
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL)
mod = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(DEVICE)
print(f"{MODEL} on {DEVICE} — {sum(p.numel() for p in mod.parameters())/1e6:.0f}M params")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


google/flan-t5-base on mps — 248M params


In [7]:
# cell 7: summariser 
from collections import Counter

GUARD_LOG = []


def _ngrams(text, n=8):
    w = re.sub(r"[^a-z0-9 ]", " ", text.lower()).split()
    return {" ".join(w[i:i + n]) for i in range(max(0, len(w) - n + 1))}


def summarise_reviews(texts, question, asin, kind, n_src=14):
    """Abstractive summary of real reviews. Returns None if the model copied."""
    texts = [t.strip() for t in texts.dropna().astype(str)][:n_src]
    if len(texts) < 3:
        GUARD_LOG.append((asin, kind, "too_few")); return None
    prompt = f"{question}\n\nCustomer reviews:\n{' '.join(texts)}"
    ids = tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    out = mod.generate(**ids, max_new_tokens=60, min_new_tokens=15, do_sample=False,
                       num_beams=4, no_repeat_ngram_size=3, length_penalty=0.8)
    text = tok.decode(out[0], skip_special_tokens=True).strip()
    source = set().union(*[_ngrams(t) for t in texts])
    if _ngrams(text) & source:            # extraction, not summarisation
        GUARD_LOG.append((asin, kind, "copied")); return None
    GUARD_LOG.append((asin, kind, "ok")); return text

In [8]:
# cell 8: render article
def _quote(q, max_chars=200):
    text = q["text"]
    if len(text) > max_chars:
        text = text[:max_chars].rsplit(" ", 1)[0] + "…"
    return f"> \"{text}\" — {q['rating']:.0f}★ review\n"


def render_article(c, n_featured=3):
    """Balanced round-up. Every figure comes from `pack`; only the optional
    'In their words' line is model-generated."""
    L = [f"# {c['cluster_name']}: what {c['n_reviews']:,} customer reviews reveal\n",
         f"Across **{c['n_products']} products** we analysed **{c['n_reviews']:,} reviews**, "
         f"averaging **{c['mean_rating']}/5**. "
         f"**{c['pct_positive']}%** are positive and **{c['pct_negative']}%** negative — "
         f"so this is a well-liked category, and the criticisms below are the exception, "
         f"not the rule.\n"]

    if c["loved_themes"]:
        L.append(f"**What buyers praise across the category:** {', '.join(c['loved_themes'][:6])}.\n")
    if c["complaint_themes"]:
        L.append(f"**What they complain about:** {', '.join(c['complaint_themes'][:6])}.\n")

    L += ["## The line-up\n",
          "| # | Product | Rating | Reviews | Positive | Would recommend |",
          "|---|---------|--------|---------|----------|-----------------|"]
    for b in c["products"]:
        rec = f"{b['recommend_rate']}%" if b["recommend_rate"] is not None else "—"
        L.append(f"| {b['rank']} | {b['name']} | {b['mean_rating']}/5 | "
                 f"{b['n_reviews']:,} | {b['pct_positive']}% | {rec} |")
    L.append("")

    for b in c["products"][:n_featured]:
        rec = (f" · **{b['recommend_rate']}%** would recommend"
               if b["recommend_rate"] is not None else "")
        L += [f"### {b['rank']}. {b['name']}\n",
              f"**{b['mean_rating']}/5** from **{b['n_reviews']:,} reviews** · "
              f"**{b['pct_positive']}%** positive{rec}\n"]

        # ---- what goes right -------------------------------------------------
        if b["loved_themes"]:
            L.append(f"**✅ What people love:** {', '.join(b['loved_themes'][:5])}.\n")
        sub = reviews[reviews["asins"] == b["asin"]]
        s = summarise_reviews(sub[sub["sentiment"] == "Positive"]["reviews.text"],
                              "Summarise what customers like about this product in two sentences.",
                              b["asin"], "pos")
        if s:
            L.append(f"*In their words:* {s}\n")
        if b["quotes_positive"]:
            L.append(_quote(b["quotes_positive"][0]))

        # ---- what goes wrong -------------------------------------------------
        if b["complaint_themes"]:
            L.append(f"**⚠️ What goes wrong** (in the {b['pct_negative']}% negative reviews): "
                     f"{', '.join(b['complaint_themes'][:5])}.\n")
        if b["quotes_negative"]:
            L.append(_quote(b["quotes_negative"][0]))

    # ---- verdict -------------------------------------------------------------
    L.append("## The verdict\n")
    top = c["products"][0]
    L.append(f"**{top['name']}** leads the category on **{top['n_reviews']:,} reviews** "
             f"at **{top['mean_rating']}/5**.\n")
    last = c["products"][-1]
    if c["data_notes"]["has_genuinely_bad_product"]:
        L.append(f"**Worth avoiding:** {last['name']} averages just **{last['mean_rating']}/5**, "
                 f"with **{last['pct_negative']}%** of its reviews negative — well outside the "
                 f"norm for this category.\n")
    else:
        L.append(f"There is no product here worth avoiding: even the lowest-rated, "
                 f"**{last['name']}**, averages **{last['mean_rating']}/5**.\n")
    return "\n".join(L)

In [9]:
# cell 9: generate & inspect
from IPython.display import Markdown, display

articles = {c["cluster_name"]: render_article(c) for c in pack["clusters"]}

print("Copy-guard results:", Counter(k for _, _, k in GUARD_LOG))
print(f"{sum(1 for _,_,k in GUARD_LOG if k=='ok')}/{len(GUARD_LOG)} summaries were genuine\n")
display(Markdown(articles[pack["clusters"][0]["cluster_name"]]))

Copy-guard results: Counter({'copied': 9, 'ok': 3})
3/12 summaries were genuine



# Fire Tablets: what 17,627 customer reviews reveal

Across **14 products** we analysed **17,627 reviews**, averaging **4.49/5**. **91.5%** are positive and **2.9%** negative — so this is a well-liked category, and the criticisms below are the exception, not the rule.

**What buyers praise across the category:** price, reading, size, read, books, fast.

**What they complain about:** slow, returned, charge, work, turn, return.

## The line-up

| # | Product | Rating | Reviews | Positive | Would recommend |
|---|---------|--------|---------|----------|-----------------|
| 1 | All New Fire Hd 8 Tablet 8 Hd Display Wifi 16 Gb Includes Special Offers Magenta | 4.59/5 | 2,814 | 94.8% | 96.3% |
| 2 | All New Fire Hd 8 Tablet 8 Hd Display Wifi 32 Gb Includes Special Offers Magenta | 4.59/5 | 158 | 94.9% | 95.6% |
| 3 | Fire Hd 10 Tablet 101 Hd Display Wifi 16 | 4.56/5 | 256 | 91.8% | 94.5% |
| 4 | Amazon Fire Hd 88 In Tablet 16 Gb Black | 4.56/5 | 270 | 93.3% | 94.4% |
| 5 | Fire Kids Edition Tablet 7 Display Wifi 16 Gb | 4.53/5 | 1,685 | 91.2% | 94.0% |
| 6 | Brand New Amazon Kindle Fire 16 Gb 7 Ips | 4.5/5 | 1,038 | 92.1% | 95.3% |
| 7 | Kindle Paperwhite Ereader White 6 High Resolution Display 300 | 4.6/5 | 30 | 96.7% | 96.7% |
| 8 | Fire Tablet 7 Display Wifi 8 Gb Includes Special Offers Magenta | 4.45/5 | 10,966 | 90.7% | 95.1% |
| 9 | Fire Tablet 7 Display Wifi 8 Gb Includes Special Offers Black | 4.42/5 | 372 | 88.7% | 93.3% |

### 1. All New Fire Hd 8 Tablet 8 Hd Display Wifi 16 Gb Includes Special Offers Magenta

**4.59/5** from **2,814 reviews** · **94.8%** positive · **96.3%** would recommend

**✅ What people love:** price, read, books, reading, movies.

> "Mother needed something since her tablet had died. Was lot easier for her to learn than her old table. She does lots more with Kindle than she did with tablet" — 5★ review

**⚠️ What goes wrong** (in the 1.9% negative reviews): returned, tried, remove, exchange, charge.

> "This is not a bad product. Amazon offers greats products but this tablet was missing features even for a basic tablet. The lack of google chrome compatibility made this tablet not right for me." — 1★ review

### 2. All New Fire Hd 8 Tablet 8 Hd Display Wifi 32 Gb Includes Special Offers Magenta

**4.59/5** from **158 reviews** · **94.9%** positive · **95.6%** would recommend

*In their words:* Love this tablet. It's very lightweight and easy to use. It has a lot of features but it doesn't support apps such as Dropbox or Google Drive.

> "This tablet is excellent considering the price. It has a very clear display, is a fairly large tablet, and has plenty of memory to store a lot of apps, music, pictures and more. Considering the…" — 5★ review

> "I had to return this product as it was not compatible with things such as Facebook and Itunes... which is what I needed it for. Great if you are into books and streaming movies!" — 1★ review

### 3. Fire Hd 10 Tablet 101 Hd Display Wifi 16

**4.56/5** from **256 reviews** · **91.8%** positive · **94.5%** would recommend

> "As I age, it's difficult for me to see the smaller screens so when this one came out .... I was delighted!!! Love playing my games!!!" — 5★ review

> "I love the kindle and have had one in one shape or another since they first came out, but I just could not take how slow this model was. I returned it and purchased a Samsung tablet." — 2★ review

## The verdict

**All New Fire Hd 8 Tablet 8 Hd Display Wifi 16 Gb Includes Special Offers Magenta** leads the category on **2,814 reviews** at **4.59/5**.

There is no product here worth avoiding: even the lowest-rated, **Fire Tablet 7 Display Wifi 8 Gb Includes Special Offers Black**, averages **4.42/5**.


In [10]:
# cell 10: save
OUT_DIR = Path("../data/processed")
(OUT_DIR / "articles.json").write_text(json.dumps(articles, indent=2, ensure_ascii=False))
OUT.write_text(json.dumps(pack, indent=2, ensure_ascii=False))   # the evidence pack, for Part 4
for name, md in articles.items():
    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    (OUT_DIR / f"article_{slug}.md").write_text(md)
print(f"saved {len(articles)} articles + evidence_pack.json")

saved 4 articles + evidence_pack.json


## The Articles

In [11]:
from IPython.display import Markdown, display

for i, (name, md) in enumerate(articles.items()):
    if i:
        display(Markdown("\n---\n"))
    display(Markdown(md))

# Fire Tablets: what 17,627 customer reviews reveal

Across **14 products** we analysed **17,627 reviews**, averaging **4.49/5**. **91.5%** are positive and **2.9%** negative — so this is a well-liked category, and the criticisms below are the exception, not the rule.

**What buyers praise across the category:** price, reading, size, read, books, fast.

**What they complain about:** slow, returned, charge, work, turn, return.

## The line-up

| # | Product | Rating | Reviews | Positive | Would recommend |
|---|---------|--------|---------|----------|-----------------|
| 1 | All New Fire Hd 8 Tablet 8 Hd Display Wifi 16 Gb Includes Special Offers Magenta | 4.59/5 | 2,814 | 94.8% | 96.3% |
| 2 | All New Fire Hd 8 Tablet 8 Hd Display Wifi 32 Gb Includes Special Offers Magenta | 4.59/5 | 158 | 94.9% | 95.6% |
| 3 | Fire Hd 10 Tablet 101 Hd Display Wifi 16 | 4.56/5 | 256 | 91.8% | 94.5% |
| 4 | Amazon Fire Hd 88 In Tablet 16 Gb Black | 4.56/5 | 270 | 93.3% | 94.4% |
| 5 | Fire Kids Edition Tablet 7 Display Wifi 16 Gb | 4.53/5 | 1,685 | 91.2% | 94.0% |
| 6 | Brand New Amazon Kindle Fire 16 Gb 7 Ips | 4.5/5 | 1,038 | 92.1% | 95.3% |
| 7 | Kindle Paperwhite Ereader White 6 High Resolution Display 300 | 4.6/5 | 30 | 96.7% | 96.7% |
| 8 | Fire Tablet 7 Display Wifi 8 Gb Includes Special Offers Magenta | 4.45/5 | 10,966 | 90.7% | 95.1% |
| 9 | Fire Tablet 7 Display Wifi 8 Gb Includes Special Offers Black | 4.42/5 | 372 | 88.7% | 93.3% |

### 1. All New Fire Hd 8 Tablet 8 Hd Display Wifi 16 Gb Includes Special Offers Magenta

**4.59/5** from **2,814 reviews** · **94.8%** positive · **96.3%** would recommend

**✅ What people love:** price, read, books, reading, movies.

> "Mother needed something since her tablet had died. Was lot easier for her to learn than her old table. She does lots more with Kindle than she did with tablet" — 5★ review

**⚠️ What goes wrong** (in the 1.9% negative reviews): returned, tried, remove, exchange, charge.

> "This is not a bad product. Amazon offers greats products but this tablet was missing features even for a basic tablet. The lack of google chrome compatibility made this tablet not right for me." — 1★ review

### 2. All New Fire Hd 8 Tablet 8 Hd Display Wifi 32 Gb Includes Special Offers Magenta

**4.59/5** from **158 reviews** · **94.9%** positive · **95.6%** would recommend

*In their words:* Love this tablet. It's very lightweight and easy to use. It has a lot of features but it doesn't support apps such as Dropbox or Google Drive.

> "This tablet is excellent considering the price. It has a very clear display, is a fairly large tablet, and has plenty of memory to store a lot of apps, music, pictures and more. Considering the…" — 5★ review

> "I had to return this product as it was not compatible with things such as Facebook and Itunes... which is what I needed it for. Great if you are into books and streaming movies!" — 1★ review

### 3. Fire Hd 10 Tablet 101 Hd Display Wifi 16

**4.56/5** from **256 reviews** · **91.8%** positive · **94.5%** would recommend

> "As I age, it's difficult for me to see the smaller screens so when this one came out .... I was delighted!!! Love playing my games!!!" — 5★ review

> "I love the kindle and have had one in one shape or another since they first came out, but I just could not take how slow this model was. I returned it and purchased a Samsung tablet." — 2★ review

## The verdict

**All New Fire Hd 8 Tablet 8 Hd Display Wifi 16 Gb Includes Special Offers Magenta** leads the category on **2,814 reviews** at **4.59/5**.

There is no product here worth avoiding: even the lowest-rated, **Fire Tablet 7 Display Wifi 8 Gb Includes Special Offers Black**, averages **4.42/5**.



---


# Echo, Fire TV & Smart Home: what 12,330 customer reviews reveal

Across **7 products** we analysed **12,330 reviews**, averaging **4.68/5**. **95.2%** are positive and **1.5%** negative — so this is a well-liked category, and the criticisms below are the exception, not the rule.

**What buyers praise across the category:** music, alexa, fun, sound, gift, family.

**What they complain about:** returned, work, money, item, return, connection.

## The line-up

| # | Product | Rating | Reviews | Positive | Would recommend |
|---|---------|--------|---------|----------|-----------------|
| 1 | Fire Tv Streaming Media Player 2015 Model | 4.71/5 | 5,056 | 95.8% | 97.1% |
| 2 | Echo White | 4.67/5 | 6,619 | 95.2% | 96.3% |
| 3 | Amazon Tap Portable Bluetooth Wifi Speaker Black | 4.53/5 | 636 | 91.7% | 93.0% |

### 1. Fire Tv Streaming Media Player 2015 Model

**4.71/5** from **5,056 reviews** · **95.8%** positive · **97.1%** would recommend

**✅ What people love:** movies, stick, cable, family, shows.

> "I bought this for one reason, and thats Kodi, its an awesome device for that, and with the sd card reader its easy to install and get up and running, highly recommended" — 5★ review

**⚠️ What goes wrong** (in the 1.4% negative reviews): returned, item, remote, roku, work.

> "Bought the fire TV excited about the 4K support. The first night it refused to load video content. I could only stream music. Now it refuses to load at all. It has completely stopped responding to…" — 1★ review

### 2. Echo White

**4.67/5** from **6,619 reviews** · **95.2%** positive · **96.3%** would recommend

**✅ What people love:** sound, music, fun, gift, day.

> "This is a great item!!! You can ask her basically anything and she will let you know the answer. Even How much wood could a woodchuck chuck if a woodchuck could chuck wood." — 5★ review

**⚠️ What goes wrong** (in the 1.3% negative reviews): google, questions, money, answer, returned.

> "Bought this as a present for my boyfriend, and we both were very disappointed. I read lots of reviews and thought it would be an awesome gift to give. Alexa is a Siri downgrade. Don't waste your…" — 1★ review

### 3. Amazon Tap Portable Bluetooth Wifi Speaker Black

**4.53/5** from **636 reviews** · **91.7%** positive · **93.0%** would recommend

> "Works great to control Alexa when she understands you and has good battery life but sound lacks bass except at low volumes. gets very loud at least. Easily connects to Bluetooth." — 4★ review

> "I spent more time trying get it functioning than I did actually keeping this product. After a couple of days and never getting it to understand or follow my commands I returned it to BB. Also it kept…" — 1★ review

## The verdict

**Fire Tv Streaming Media Player 2015 Model** leads the category on **5,056 reviews** at **4.71/5**.

There is no product here worth avoiding: even the lowest-rated, **Amazon Tap Portable Bluetooth Wifi Speaker Black**, averages **4.53/5**.



---


# Kindle E-Readers: what 4,092 customer reviews reveal

Across **6 products** we analysed **4,092 reviews**, averaging **4.75/5**. **96.9%** are positive and **1.0%** negative — so this is a well-liked category, and the criticisms below are the exception, not the rule.

**What buyers praise across the category:** reading, battery, sun, wife, able, life.

**What they complain about:** tried, work, customer, service, computer, usb.

## The line-up

| # | Product | Rating | Reviews | Positive | Would recommend |
|---|---------|--------|---------|----------|-----------------|
| 1 | Amazon Kindle Paperwhite Ebook Reader 4 Gb 6 Monochrome | 4.77/5 | 3,176 | 97.3% | 98.2% |
| 2 | Kindle Voyage Ereader 6 High Resolution Display 300 Ppi | 4.73/5 | 580 | 96.7% | 97.8% |
| 3 | Amazon Kindle Voyage 4 Gb Wifi 3 Gb Lack | 4.86/5 | 51 | 100.0% | 100.0% |
| 4 | Kindle Oasis Ereader With Leather Charging Cover Mer Lot | 4.61/5 | 67 | 94.0% | 94.0% |
| 5 | All New Kindle Ereader Black 6 Glare Free Touch | 4.53/5 | 212 | 91.5% | 93.4% |

### 1. Amazon Kindle Paperwhite Ebook Reader 4 Gb 6 Monochrome

**4.77/5** from **3,176 reviews** · **97.3%** positive · **98.2%** would recommend

**✅ What people love:** reading, paperwhite, battery, light, life.

> "I just purchased it yesterday, and am no expert by far, but setup was a breeze. I also took it to the beach today and found that it was like reading a piece of paper. Amazing." — 5★ review

**⚠️ What goes wrong** (in the 0.9% negative reviews): work, tried, computer, books library, library.

> "Paper white does not allow you do use books from the library as you cannot load apps. Only books from Amazon are available to read." — 1★ review

### 2. Kindle Voyage Ereader 6 High Resolution Display 300 Ppi

**4.73/5** from **580 reviews** · **96.7%** positive · **97.8%** would recommend

> "If you already have a Paperwhite then there is no need to upgrade. But if you are in the market this has an excellent display and quick page turns. The reason to buy a dedicated e-reader is the 3…" — 4★ review

> "very good produ ct and service will defintely refer to a freind." — 1★ review

### 3. Amazon Kindle Voyage 4 Gb Wifi 3 Gb Lack

**4.86/5** from **51 reviews** · **100.0%** positive · **100.0%** would recommend

> "I bought the Voyage kindle as an anniversary present for my wife and she LOVES it. The new dim light feature and matted glass screen are pretty amazing and work like a charm. She's able to read in…" — 5★ review

## The verdict

**Amazon Kindle Paperwhite Ebook Reader 4 Gb 6 Monochrome** leads the category on **3,176 reviews** at **4.77/5**.

There is no product here worth avoiding: even the lowest-rated, **All New Kindle Ereader Black 6 Glare Free Touch**, averages **4.53/5**.



---


# Accessories & Cables: what 560 customer reviews reveal

Across **10 products** we analysed **560 reviews**, averaging **4.29/5**. **81.8%** are positive and **12.7%** negative — so this is a well-liked category, and the criticisms below are the exception, not the rule.

**What buyers praise across the category:** charges, needed, new, perfectly, thanks, item.

**What they complain about:** charger, included, expensive, charge, money, overpriced.

## The line-up

| # | Product | Rating | Reviews | Positive | Would recommend |
|---|---------|--------|---------|----------|-----------------|
| 1 | Amazon 5W Usb Official Oem Charger Power Adapter For Fire Tablets Kindle Ereaders [B01J2G4VBG] | 4.44/5 | 401 | 85.8% | — |
| 2 | Amazon Echo Fire Tv Power Adapter | 4.5/5 | 16 | 87.5% | — |
| 3 | Amazon 9W Power Fast Official Oem Usb Charger Power | 4.21/5 | 73 | 79.5% | 94.4% |
| 4 | Amazon 5W Usb Official Oem Charger Power Adapter For Fire Tablets Kindle Ereaders [B00QL1ZN3G] | 3.07/5 | 15 | 53.3% | — |

### 1. Amazon 5W Usb Official Oem Charger Power Adapter For Fire Tablets Kindle Ereaders [B01J2G4VBG]

**4.44/5** from **401 reviews** · **85.8%** positive

**✅ What people love:** worked, charges, perfectly, item, thanks.

> "Very expensive charger. I think the kindle should come with it as does all other tablets, including the fire, which is way cheaper and includes the charger for just 50. It is a pretty large charger…" — 4★ review

**⚠️ What goes wrong** (in the 9.2% negative reviews): expensive, included, charge, adapter, cost.

> "I ordered the Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders however, what I received is nothing like what is pictured. I received a charger cord that…" — 2★ review

### 2. Amazon Echo Fire Tv Power Adapter

**4.5/5** from **16 reviews** · **87.5%** positive

*In their words:* Amazon gave me a new power cord at a much cheaper price.

> "These are made for the Echo and as I use my Echo in a couple ends of the house so it is convenient to have the two of them" — 4★ review

> "Bought Amazon Fire TV 2/16/15, this is NOT the replacement cord for my model.Be aware of what you are purchasing --- this is NOT a fit for the older Fire models.Alternate, non Amazon, power cords are…" — 2★ review

### 3. Amazon 9W Power Fast Official Oem Usb Charger Power

**4.21/5** from **73 reviews** · **79.5%** positive · **94.4%** would recommend

*In their words:* Great charger for my kindle 7 tablet. Great price for what you get.

> "got this for my kindle 7 tablet . Does an excellent job charging the kindle fire 7 a lot faster than the one it came with the kindle fire" — 5★ review

> "I just used this after receiving it a few days. I tried a couple times to use it, but my Kindle registered connected to a low power charger. Too expensive for this to happen. I've never written a…" — 1★ review

## The verdict

**Amazon 5W Usb Official Oem Charger Power Adapter For Fire Tablets Kindle Ereaders [B01J2G4VBG]** leads the category on **401 reviews** at **4.44/5**.

**Worth avoiding:** Amazon 5W Usb Official Oem Charger Power Adapter For Fire Tablets Kindle Ereaders [B00QL1ZN3G] averages just **3.07/5**, with **40.0%** of its reviews negative — well outside the norm for this category.
